# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [2]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

root_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "notebooks").exists()),
    Path.cwd()
)

OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据: {DATA_PATH}")
print(f"项目输出目录: {OUTPUT_DIR}")
print(f"原始数据形状: {raw_df.shape}")
raw_df.head()


原始数据: ..\data\E Commerce Dataset.xlsx
项目输出目录: C:\Users\ppp\Desktop\实习\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project
原始数据形状: (5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [3]:
# 在此写下你的答案：
# 1.每条记录代表一位电商平台用户，包含该用户的ID、使用时长、登陆偏好、城市等级、消费行为、流失状态等一系列个人与业务相关的特征信息
# 2.目标变量是Churn（客户流失）列
# 3.CustomerID是无数值大小意义的用户唯一标识，仅用于身份区分，作为连续数值分析会引入无效虚假规律、干扰模型/分析结果，因此不能当作普通联系数值使用

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [4]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    # TODO：返回一个 DataFrame，至少包含：
    # 数据类型、缺失数量、缺失比例(%)、唯一值数量
    # 1. 字段数据类型
    dtype_series = data.dtypes
    # 2. 缺失数量
    miss_cnt = data.isna().sum()
    # 3. 缺失百分比，保留2位小数
    miss_pct = (data.isna().mean() * 100).round(2)
    # 4. 唯一值数量
    unique_cnt = data.nunique()
    # 拼接成报告DataFrame
    report = pd.DataFrame({
        "数据类型": dtype_series,
        "缺失数量": miss_cnt,
        "缺失比例(%)": miss_pct,
        "唯一值数量": unique_cnt
    })
    return report


# TODO：生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)

,数据类型,缺失数量,缺失比例(%),唯一值数量
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,str,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,str,0,0.00,7
Gender,str,0,0.00,2
HourSpendOnApp,float64,255,4.53,6
NumberOfDeviceRegistered,int64,0,0.00,6


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [5]:
# TODO：完成项目初始审计
print("完全重复行数：", raw_df.duplicated().sum())
print("CustomerID 重复数量：", raw_df["CustomerID"].duplicated().sum())
print("\nChurn 流失标签频数统计：")
print(raw_df["Churn"].value_counts())
churn_cnt = raw_df["Churn"].sum()
total = len(raw_df)
churn_rate = (churn_cnt / total) * 100
print(f"流失率：{churn_rate:.2f}%")
#
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n=====字段:{col}频次分布=====")
    print(raw_df[col].value_counts())

完全重复行数： 0
CustomerID 重复数量： 0

Churn 流失标签频数统计：
Churn
0    4682
1     948
Name: count, dtype: int64
流失率：16.84%

=====字段:PreferredLoginDevice频次分布=====
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

=====字段:PreferredPaymentMode频次分布=====
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

=====字段:PreferedOrderCat频次分布=====
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [6]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [7]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据，不修改原始输入数据，返回清洗后数据与处理日志。

    参数:
        data: 原始用户行为 DataFrame

    返回:
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 清洗过程日志 DataFrame，包含每一步处理的明细
    """
    # 1. 复制原始数据，避免直接修改传入的原始DataFrame
    df = data.copy()
    log_entries = []  # 初始化日志列表


    # ========== 步骤1：删除完全重复行 ==========
    before_total = len(df)
    df = df.drop_duplicates(keep="first")  # 保留第一条，删除后续重复行
    after_total = len(df)
    affected_rows = before_total - after_total

    # 记录日志
    log_entries.append({
        "处理步骤": "删除完全重复行",
        "处理规则": "删除整行所有字段完全一致的重复记录，仅保留第一条",
        "处理前记录数": before_total,
        "处理后记录数": after_total,
        "影响记录数": affected_rows
    })

    # ========== 步骤2：数值列中位数填充缺失值 ==========
    for col in NUMERIC_MISSING_COLS:
        before_miss = df[col].isna().sum()  # 填充前该列缺失数量
        median_fill = df[col].median()     # 计算全局中位数（不按Churn分组）
        df[col] = df[col].fillna(median_fill)
        after_miss = df[col].isna().sum()   # 填充后该列缺失数量
        affected_miss = before_miss - after_miss

        # 记录每一列的填充日志
        log_entries.append({
            "处理步骤": "数值列缺失值填充",
            "处理规则": f"对列【{col}】使用全局中位数填充缺失值",
            "处理前记录数": before_miss,
            "处理后记录数": after_miss,
            "影响记录数": affected_miss
        })

    # ========== 步骤3：类别字段同义值标准化 ==========
    for col, replace_dict in CATEGORY_MAPPINGS.items():
        for old_val, new_val in replace_dict.items():
            before_cnt = (df[col] == old_val).sum()  # 替换前该值的数量
            df[col] = df[col].replace(old_val, new_val)
            after_cnt = (df[col] == old_val).sum()   # 替换后该值的剩余数量
            affected_cnt = before_cnt - after_cnt

            # 记录每一条映射规则的日志
            log_entries.append({
                "处理步骤": "类别字段标准化",
                "处理规则": f"将列【{col}】中的「{old_val}」统一替换为「{new_val}」",
                "处理前记录数": before_cnt,
                "处理后记录数": after_cnt,
                "影响记录数": affected_cnt
            })

    # ========== 步骤4：数据类型转换 ==========
    # 对Churn、Complain字段转为整数类型
    for col in ["Churn", "Complain"]:
        before_type = str(df[col].dtype)
        df[col] = df[col].astype(int)
        after_type = str(df[col].dtype)

        # 记录类型转换日志
        log_entries.append({
            "处理步骤": "数据类型转换",
            "处理规则": f"将列【{col}】从{before_type}转换为{after_type}",
            "处理前记录数": len(df),
            "处理后记录数": len(df),
            "影响记录数": 0
        })

    # ========== 生成日志DataFrame并返回 ==========
    cleaning_log = pd.DataFrame(log_entries)
    return df, cleaning_log

### 任务 3：运行清洗函数并查看日志

In [8]:
# TODO：执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

display(cleaning_log)
cleaned_df.head()

,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,删除完全重复行,删除整行所有字段完全一致的重复记录，仅保留第一条,5630,5630,0
1,数值列缺失值填充,对列【Tenure】使用全局中位数填充缺失值,264,0,264
2,数值列缺失值填充,对列【WarehouseToHome】使用全局中位数填充缺失值,251,0,251
3,数值列缺失值填充,对列【HourSpendOnApp】使用全局中位数填充缺失值,255,0,255
4,数值列缺失值填充,对列【OrderAmountHikeFromlastYear】使用全局中位数填充缺失值,265,0,265
5,数值列缺失值填充,对列【CouponUsed】使用全局中位数填充缺失值,256,0,256
6,数值列缺失值填充,对列【OrderCount】使用全局中位数填充缺失值,258,0,258
7,数值列缺失值填充,对列【DaySinceLastOrder】使用全局中位数填充缺失值,307,0,307
8,类别字段标准化,将列【PreferredLoginDevice】中的「Phone」统一替换为「Mobile ...,1231,0,1231
9,类别字段标准化,将列【PreferredPaymentMode】中的「COD」统一替换为「Cash on D...,365,0,365


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [9]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# 1. 定义分箱区间与标签（通用业务分层区间）
tenure_bins = [0, 6, 12, 24, 999]
tenure_labels = ["0-6个月", "7-12个月", "13-24个月", "24个月以上"]

# 新建分层列 TenureGroup
cleaned_df["TenureGroup"] = pd.cut(
    cleaned_df["Tenure"],
    bins=tenure_bins,
    labels=tenure_labels,
    right=True,
    include_lowest=True
)
cleaned_df["IsMobileLogin"] = (cleaned_df["PreferredLoginDevice"] == "Mobile Phone").astype(int)
# 待检测异常的字段列表
outlier_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_report_list = []

for col_name in outlier_cols:
    res = iqr_outlier_summary(cleaned_df[col_name])
    res["字段名"] = col_name
    outlier_report_list.append(res)

# 转为标准DataFrame报表
outlier_report = pd.DataFrame(outlier_report_list)
# 调整列顺序方便查看
outlier_report = outlier_report[["字段名","Q1","Q3","下限","上限","候选异常值数量"]]

# 打印查看异常报告
print("===== IQR候选异常值汇总报告 =====")
display(outlier_report)


===== IQR候选异常值汇总报告 =====


,字段名,Q1,Q3,下限,上限,候选异常值数量
0,WarehouseToHome,9.00,20.00,-7.50,36.50,2
1,OrderCount,1.00,3.00,-2.00,6.00,703
2,CashbackAmount,145.77,196.39,69.84,272.33,438


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [10]:
# TODO：完成业务规则检查
# business_rule_report = pd.DataFrame({
#     "规则": [...],
#     "不合规记录数": [...]
# })
# display(business_rule_report)
#
# 处理结论：
# 逐条统计各业务规则的不合规记录数
rule1_count = (cleaned_df["Tenure"] < 0).sum()          # 使用时长小于0
rule2_count = (cleaned_df["WarehouseToHome"] < 0).sum()# 仓库距离小于0
rule3_count = (cleaned_df["OrderCount"] <= 0).sum()     # 订单数小于或等于0
rule4_count = (cleaned_df["CashbackAmount"] < 0).sum() # 返现金额小于0

# 构建业务规则检查报表
business_rule_report = pd.DataFrame({
    "规则": [
        "使用时长小于 0",
        "仓库距离小于 0",
        "订单数小于或等于 0",
        "返现金额小于 0"
    ],
    "不合规记录数": [rule1_count, rule2_count, rule3_count, rule4_count]
})

# 展示报表
display(business_rule_report)

# 输出处理结论
print("===== 处理结论 =====")
total_invalid = business_rule_report["不合规记录数"].sum()
if total_invalid == 0:
    print("1. 所有业务规则检查项的不合规记录数均为0，数据集完全符合本次设定的业务数值约束；")
    print("2. 无需针对这类违规数据做删除、修正操作，可直接用于后续分析建模；")
else:
    print(f"1. 本次累计发现{total_invalid}条违反业务规则的记录，明细见上方报表；")
    print("2. 按照项目之前要求，仅做记录留存，不自动删除违规数据；")
    print("3. 后续分析时可根据业务需求决定是否剔除这部分异常样本。")


,规则,不合规记录数
0,使用时长小于 0,0
1,仓库距离小于 0,0
2,订单数小于或等于 0,0
3,返现金额小于 0,0


===== 处理结论 =====
1. 所有业务规则检查项的不合规记录数均为0，数据集完全符合本次设定的业务数值约束；
2. 无需针对这类违规数据做删除、修正操作，可直接用于后续分析建模；


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [11]:
# TODO: 完成最终验收
quality_after = build_quality_report(cleaned_df)

assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique()
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique()
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns)
print("全部验收，校验通过，清洗流程合规")
print("\n=====清洗前数据质量报告====")
display(quality_before)
print("\n=====清洗后数据质量报告====")
display(quality_after)

# TODO：导出下列文件，使用 utf-8-sig 编码；统一关闭索引导出 index=False
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=False, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=False, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
# 修改文件名与校验脚本要求完全一致
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")

print("✅ 项目所有交付文件已全部导出，文件路径清单：")
file_list = [
    "data_quality_before.csv",
    "data_quality_after.csv",
    "cleaning_log.csv",
    "ecommerce_customer_cleaned.csv",
    "outlier_report.csv",
    "business_rule_report.csv"
]

for fname in file_list:
    full_path = (OUTPUT_DIR / fname).resolve()
    print(f"• {full_path}")


全部验收，校验通过，清洗流程合规

=====清洗前数据质量报告====


,数据类型,缺失数量,缺失比例(%),唯一值数量
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,str,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,str,0,0.00,7
Gender,str,0,0.00,2
HourSpendOnApp,float64,255,4.53,6
NumberOfDeviceRegistered,int64,0,0.00,6



=====清洗后数据质量报告====


,数据类型,缺失数量,缺失比例(%),唯一值数量
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,0,0.00,36
PreferredLoginDevice,str,0,0.00,2
CityTier,int64,0,0.00,3
WarehouseToHome,float64,0,0.00,34
PreferredPaymentMode,str,0,0.00,5
Gender,str,0,0.00,2
HourSpendOnApp,float64,0,0.00,6
NumberOfDeviceRegistered,int64,0,0.00,6


✅ 项目所有交付文件已全部导出，文件路径清单：
• C:\Users\ppp\Desktop\实习\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\data_quality_before.csv
• C:\Users\ppp\Desktop\实习\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\data_quality_after.csv
• C:\Users\ppp\Desktop\实习\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\cleaning_log.csv
• C:\Users\ppp\Desktop\实习\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\ecommerce_customer_cleaned.csv
• C:\Users\ppp\Desktop\实习\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\outlier_report.csv
• C:\Users\ppp\Desktop\实习\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\business_rule_report.csv


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

In [12]:
#1.发现的质量问题：数值字段存在缺失值；登陆设备、支付方式存在类别命名不统一问题；部分指标出现违背业务逻辑的负数；部分字段存在IQR判定的候选异常值
#2.处理策略：数值缺失用全局中位数填充；类别做统一归依合并；候选异常值、业务违规负数仅记录留存，不擅自删除
#3.适配后续分析原因：缺失值清零、类别统一规范、新增分层与业务约束均满足分析要求
#4.待业务确认规则：候选异常值、业务负向违规数据是否剔除的阈值标准；分层区间划分是否贴合实际运营场景